Using UCI Online Retail Sales Dataset
- Will have to drive all features


In [317]:
import pandas as pd
import numpy as np
df = pd.read_csv("Online Retail Cleaned.csv")

In [318]:
info_org = df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 535283 entries, 0 to 535282
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   InvoiceNo       535283 non-null  object 
 1   StockCode       535283 non-null  object 
 2   Description     535283 non-null  object 
 3   Quantity        535283 non-null  int64  
 4   InvoiceDate     535283 non-null  object 
 5   UnitPrice       535283 non-null  float64
 6   CustomerID      535283 non-null  int64  
 7   Country         534841 non-null  object 
 8   IsCancellation  535283 non-null  bool   
 9   IsGuest         535283 non-null  bool   
 10  IsNonProduct    535283 non-null  bool   
 11  BaseStockCode   532496 non-null  float64
 12  IsVariant       535283 non-null  bool   
dtypes: bool(4), float64(2), int64(2), object(5)
memory usage: 38.8+ MB


In [319]:
df['TransactionValue'] = df['UnitPrice'] * df['Quantity']
print(df.columns)
print(df.isna().sum())

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'IsCancellation', 'IsGuest',
       'IsNonProduct', 'BaseStockCode', 'IsVariant', 'TransactionValue'],
      dtype='object')
InvoiceNo              0
StockCode              0
Description            0
Quantity               0
InvoiceDate            0
UnitPrice              0
CustomerID             0
Country              442
IsCancellation         0
IsGuest                0
IsNonProduct           0
BaseStockCode       2787
IsVariant              0
TransactionValue       0
dtype: int64


In [320]:
df[(~df['InvoiceNo'].str.isnumeric()) & (~df['IsCancellation'])]
df[(~df['StockCode'].str.isnumeric()) & (~df['IsNonProduct'])]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancellation,IsGuest,IsNonProduct,BaseStockCode,IsVariant,TransactionValue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,False,False,False,85123.0,True,15.30
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,False,False,False,84406.0,True,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,False,84029.0,True,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,False,84029.0,True,20.34
49,536373,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 09:02:00,2.55,17850,United Kingdom,False,False,False,85123.0,True,15.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535152,581578,84997C,CHILDRENS CUTLERY POLKADOT BLUE,8,2011-12-09 12:16:00,4.15,12713,Germany,False,False,False,84997.0,True,33.20
535183,581579,85099C,JUMBO BAG BAROQUE BLACK WHITE,10,2011-12-09 12:19:00,1.79,17581,United Kingdom,False,False,False,85099.0,True,17.90
535212,581580,84993A,75 GREEN PETIT FOUR CASES,2,2011-12-09 12:20:00,0.42,12748,United Kingdom,False,False,False,84993.0,True,0.84
535218,581580,85049A,TRADITIONAL CHRISTMAS RIBBONS,1,2011-12-09 12:20:00,1.25,12748,United Kingdom,False,False,False,85049.0,True,1.25


In [321]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['BaseStockCode'] = df['BaseStockCode'].astype('Int64')

In [322]:
df = df[~df['IsCancellation']]
df = df[~df['IsNonProduct']]
df = df[~df['IsGuest']]

In [323]:
# df.info()
df.isna().sum()

InvoiceNo             0
StockCode             0
Description           0
Quantity              0
InvoiceDate           0
UnitPrice             0
CustomerID            0
Country             241
IsCancellation        0
IsGuest               0
IsNonProduct          0
BaseStockCode         0
IsVariant             0
TransactionValue      0
dtype: int64

In [324]:
transactions_start_date, transactions_end_date = df['InvoiceDate'].min(), df['InvoiceDate'].max()
Total_months = 13

Start and End date of transactions  
Timestamp('2010-12-01 08:26:00')   
Timestamp('2011-12-09 12:50:00')  

The data has 13 months 2010-12 to 2011-12

In [325]:
cust_grp = df.groupby('CustomerID')
cust_transactions = cust_grp.agg(
    TransactionCount = ('InvoiceNo', 'nunique'),
    FirstTransaction = ('InvoiceDate', 'min'),
    RecentTransaction = ('InvoiceDate', 'max')
).sort_values(by='TransactionCount', ascending=False)
cust_transactions
cust_transactions[(cust_transactions['TransactionCount']>=10)]

,TransactionCount,FirstTransaction,RecentTransaction
CustomerID,,,
12748,206,2010-12-01 12:48:00,2011-12-09 12:20:00
14911,199,2010-12-01 14:05:00,2011-12-08 15:54:00
17841,124,2010-12-01 14:41:00,2011-12-08 12:07:00
13089,97,2010-12-05 10:27:00,2011-12-07 09:02:00
15311,91,2010-12-01 09:41:00,2011-12-09 12:00:00
...,...,...,...
16985,10,2010-12-16 10:38:00,2011-11-22 13:33:00
14243,10,2010-12-09 08:34:00,2011-12-01 11:12:00
13769,10,2010-12-07 13:28:00,2011-12-07 15:08:00


- Total Customers: 4335 
- Customers with >10 transactions: 387  
The Retail Data will be used for churn predictions customers with low transactions will affect the quality of training dataset.

In [326]:
# Old Cutomers with low transactions
old_cust_low_tran = cust_transactions[(cust_transactions['TransactionCount']<10) & (cust_transactions['RecentTransaction'] > pd.Timestamp('2011-09-01')) & (cust_transactions['FirstTransaction'] < pd.Timestamp('2010-12-31'))].index
high_tran_cust = cust_transactions[cust_transactions['TransactionCount'] >= 10].index

In [327]:
high_tran_cust.size + old_cust_low_tran.size

748

- Will use transactions of only 748 customers

Possible Features:
- RFM features
- Customer Segments/ behavioural features

In [328]:
df = df[(df['CustomerID'].isin(old_cust_low_tran)) | (df['CustomerID'].isin(high_tran_cust)) ]
df = df[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country','BaseStockCode', 'IsVariant', 'TransactionValue']]
df["TransactionMonth"] = df['InvoiceDate'].dt.month
df['TransactionYear'] = df['InvoiceDate'].dt.year
df['CustomerID'].nunique()

748

In [329]:
filtered_Customer_Group = df.groupby('CustomerID')
rfm_features = filtered_Customer_Group.agg(
    FirstTransaction = ('InvoiceDate', 'min'),
    RecentTransaction = ('InvoiceDate', 'max'),
    TotalTransactions = ('InvoiceNo','nunique'),
    AvgTransactionValue = ('TransactionValue','mean'),
    MaxTransactionValue = ('TransactionValue','max'),
    MinTransactionValue = ('TransactionValue','min'),
    Revenue = ('TransactionValue','sum')
)
rfm_features['CustomerLife(Days)'] = (rfm_features['RecentTransaction'] - rfm_features['FirstTransaction']).dt.days
rfm_features['DaysSinceLastTransaction'] = (transactions_end_date - rfm_features['RecentTransaction']).dt.days
rfm_features['AvgDaysBetweenTransactions'] = rfm_features['CustomerLife(Days)'] / (rfm_features['TotalTransactions'] -1)
monthly_Transactions = df.groupby(['CustomerID', 'TransactionYear', 'TransactionMonth']).agg(
    TransactionsPerMonth = ('InvoiceNo','nunique')
).reset_index().groupby('CustomerID').agg(
    AvgTransactionsPerMonth = ('TransactionsPerMonth','mean')
)
rfm_features['AvgTransactionsPerMonth'] = monthly_Transactions['AvgTransactionsPerMonth']
rfm_features.sort_values(by='CustomerLife(Days)', ascending=True)

,FirstTransaction,RecentTransaction,TotalTransactions,AvgTransactionValue,MaxTransactionValue,MinTransactionValue,Revenue,CustomerLife(Days),DaysSinceLastTransaction,AvgDaysBetweenTransactions,AvgTransactionsPerMonth
CustomerID,,,,,,,,,,,
17850,2010-12-01 08:26:00,2010-12-02 15:27:00,34,18.152222,107.25,6.36,5391.21,1,371,0.030303,34.000000
18073,2011-05-26 19:38:00,2011-08-17 14:02:00,12,53.608611,305.28,7.80,3859.82,82,113,7.454545,3.000000
14096,2011-08-30 10:49:00,2011-12-05 17:17:00,17,10.453078,264.32,0.42,53258.43,97,3,6.062500,3.400000
16764,2011-07-19 17:17:00,2011-12-06 11:41:00,13,3.987071,33.15,0.29,2790.95,139,3,11.583333,2.166667
15235,2010-12-01 17:22:00,2011-05-06 09:44:00,12,15.716853,35.40,5.04,2247.51,155,217,14.090909,2.400000
...,...,...,...,...,...,...,...,...,...,...,...
16210,2010-12-01 12:27:00,2011-12-08 12:36:00,17,172.756557,3254.40,6.72,21076.30,372,1,23.250000,2.125000
14606,2010-12-01 16:57:00,2011-12-08 19:28:00,90,4.460041,139.30,0.12,11926.15,372,0,4.179775,6.923077
12748,2010-12-01 12:48:00,2011-12-09 12:20:00,206,7.198267,528.00,0.06,31650.78,372,0,1.814634,15.846154


11 RFM features: 
- 2 R, 5 F, 4 M

In [330]:
rfm_features.columns
rfm_features = rfm_features[['FirstTransaction', 'RecentTransaction',  # Recency 
        'TotalTransactions', 'CustomerLife(Days)', 'DaysSinceLastTransaction',
        'AvgDaysBetweenTransactions', 'AvgTransactionsPerMonth',     # Frequency
        'AvgTransactionValue', 'MaxTransactionValue', 'MinTransactionValue',
        'Revenue' ]]   # Monetary
rfm_features

,FirstTransaction,RecentTransaction,TotalTransactions,CustomerLife(Days),DaysSinceLastTransaction,AvgDaysBetweenTransactions,AvgTransactionsPerMonth,AvgTransactionValue,MaxTransactionValue,MinTransactionValue,Revenue
CustomerID,,,,,,,,,,,
12347,2010-12-07 14:57:00,2011-12-07 15:52:00,7,365,1,60.833333,1.000000,23.681319,249.6,5.04,4310.00
12348,2010-12-16 19:09:00,2011-09-25 13:13:00,4,282,74,94.000000,1.000000,53.231111,150.0,13.20,1437.24
12362,2011-02-17 10:30:00,2011-12-06 15:40:00,10,292,2,32.444444,1.250000,18.504805,59.9,2.88,4737.23
12370,2010-12-14 12:58:00,2011-10-19 14:51:00,4,309,50,103.000000,1.333333,20.739030,163.2,1.25,3421.94
12395,2010-12-03 16:35:00,2011-11-20 15:21:00,12,351,18,31.909091,1.500000,18.629375,76.5,2.50,2682.63
...,...,...,...,...,...,...,...,...,...,...,...
18229,2010-12-01 16:25:00,2011-11-28 09:48:00,20,361,11,19.000000,1.666667,44.371341,272.0,8.50,7276.90
18241,2011-05-06 09:22:00,2011-11-30 12:10:00,17,208,9,13.000000,2.428571,19.933558,59.4,5.04,2073.09
18245,2010-12-19 14:58:00,2011-12-02 14:48:00,7,347,6,57.833333,1.400000,14.668914,39.8,4.56,2567.06


In [331]:
df.columns
df['IsWeekend'] = (df['InvoiceDate'].dt.dayofweek > 4)

In [332]:
Behavioral_feature = df.groupby(['CustomerID','TransactionYear','TransactionMonth']).agg(
    TransactionMonths = ('TransactionMonth','nunique')
).reset_index().groupby('CustomerID').agg(
    ActiveMonths = ('TransactionMonths','count')
)
Behavioral_feature['InactiveMonths'] = Total_months - Behavioral_feature['ActiveMonths']
weekend = df[df['IsWeekend']].groupby('CustomerID').agg(
    NoOfWeekendTransactions = ('InvoiceNo','nunique')
)
Behavioral_feature = Behavioral_feature.merge(weekend, how='left', on='CustomerID')
weekday = df[~df['IsWeekend']].groupby('CustomerID').agg(
    NoOfWeekdayTransactions = ('InvoiceNo','nunique')
)
Behavioral_feature = Behavioral_feature.merge(weekday, how='left', on='CustomerID')
Behavioral_feature = Behavioral_feature.fillna(0)
x = df.groupby(['CustomerID','InvoiceNo']).agg(
    date=('InvoiceDate', 'min')
).sort_values(by='date')
x['gap'] = x.groupby('CustomerID')['date'].diff().dt.days.astype('Int64')
max_gap = (x.groupby('CustomerID')['gap'].max().reset_index())
Behavioral_feature = Behavioral_feature.merge(max_gap, how='left', on='CustomerID')
Behavioral_feature = Behavioral_feature.rename(columns={'gap': 'MaxGapBetweentransactions'})
Behavioral_feature

,CustomerID,ActiveMonths,InactiveMonths,NoOfWeekendTransactions,NoOfWeekdayTransactions,MaxGapBetweentransactions
0,12347,7,6,0.0,7.0,90
1,12348,4,9,1.0,3.0,173
2,12362,8,5,0.0,10.0,70
3,12370,3,10,0.0,4.0,223
4,12395,8,5,1.0,11.0,55
...,...,...,...,...,...,...
743,18229,12,1,3.0,17.0,36
744,18241,7,6,0.0,17.0,41
745,18245,5,8,1.0,6.0,142
746,18259,3,10,0.0,3.0,281


In [333]:
data = rfm_features.merge(Behavioral_feature, how='inner', on='CustomerID')
data

,CustomerID,FirstTransaction,RecentTransaction,TotalTransactions,CustomerLife(Days),DaysSinceLastTransaction,AvgDaysBetweenTransactions,AvgTransactionsPerMonth,AvgTransactionValue,MaxTransactionValue,MinTransactionValue,Revenue,ActiveMonths,InactiveMonths,NoOfWeekendTransactions,NoOfWeekdayTransactions,MaxGapBetweentransactions
0,12347,2010-12-07 14:57:00,2011-12-07 15:52:00,7,365,1,60.833333,1.000000,23.681319,249.6,5.04,4310.00,7,6,0.0,7.0,90
1,12348,2010-12-16 19:09:00,2011-09-25 13:13:00,4,282,74,94.000000,1.000000,53.231111,150.0,13.20,1437.24,4,9,1.0,3.0,173
2,12362,2011-02-17 10:30:00,2011-12-06 15:40:00,10,292,2,32.444444,1.250000,18.504805,59.9,2.88,4737.23,8,5,0.0,10.0,70
3,12370,2010-12-14 12:58:00,2011-10-19 14:51:00,4,309,50,103.000000,1.333333,20.739030,163.2,1.25,3421.94,3,10,0.0,4.0,223
4,12395,2010-12-03 16:35:00,2011-11-20 15:21:00,12,351,18,31.909091,1.500000,18.629375,76.5,2.50,2682.63,8,5,1.0,11.0,55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
743,18229,2010-12-01 16:25:00,2011-11-28 09:48:00,20,361,11,19.000000,1.666667,44.371341,272.0,8.50,7276.90,12,1,3.0,17.0,36
744,18241,2011-05-06 09:22:00,2011-11-30 12:10:00,17,208,9,13.000000,2.428571,19.933558,59.4,5.04,2073.09,7,6,0.0,17.0,41
745,18245,2010-12-19 14:58:00,2011-12-02 14:48:00,7,347,6,57.833333,1.400000,14.668914,39.8,4.56,2567.06,5,8,1.0,6.0,142
746,18259,2010-12-08 13:38:00,2011-11-15 12:34:00,3,341,24,170.500000,1.000000,55.680952,135.0,12.60,2338.60,3,10,0.0,3.0,281


Basic Churn Condition: No Transaction in last 90 days

In [334]:
data['Churner'] = data['DaysSinceLastTransaction'] > 90
data['FirstTransaction'] = (data['FirstTransaction'] - transactions_start_date).dt.days.astype('Int64')
data = data.rename(columns={'FirstTransaction':'TransactionAfterStartDate(Days)'})
data.head()

,CustomerID,TransactionAfterStartDate(Days),RecentTransaction,TotalTransactions,CustomerLife(Days),DaysSinceLastTransaction,AvgDaysBetweenTransactions,AvgTransactionsPerMonth,AvgTransactionValue,MaxTransactionValue,MinTransactionValue,Revenue,ActiveMonths,InactiveMonths,NoOfWeekendTransactions,NoOfWeekdayTransactions,MaxGapBetweentransactions,Churner
0,12347,6,2011-12-07 15:52:00,7,365,1,60.833333,1.000000,23.681319,249.6,5.04,4310.00,7,6,0.0,7.0,90,False
1,12348,15,2011-09-25 13:13:00,4,282,74,94.000000,1.000000,53.231111,150.0,13.20,1437.24,4,9,1.0,3.0,173,False
2,12362,78,2011-12-06 15:40:00,10,292,2,32.444444,1.250000,18.504805,59.9,2.88,4737.23,8,5,0.0,10.0,70,False
3,12370,13,2011-10-19 14:51:00,4,309,50,103.000000,1.333333,20.739030,163.2,1.25,3421.94,3,10,0.0,4.0,223,False
4,12395,2,2011-11-20 15:21:00,12,351,18,31.909091,1.500000,18.629375,76.5,2.50,2682.63,8,5,1.0,11.0,55,False


Changes:
- First Transaction into days (Difference between start date and first transaction)
- Drop Recent Transaction We already have days since Last Transaction

In [335]:
data.to_csv("ProcessedCustomerData.csv",index=False)